In [1]:
from __future__ import annotations

from enum import Enum
import errno
import sys
import time
from pathlib import Path
from typing import Iterable, Iterator, Optional
import logging
import os
import re
from importlib import reload

from debugpy.common.log import LEVELS

In [2]:
from identifier import Id, CompositeId
from meta import Run, Dir, File
from db import Db

In [3]:
import identifier
reload(identifier)
from identifier import Id, CompositeId

In [4]:
import db
reload(db)
from db import Db

In [3]:
db_name = 'test-db-for-db'
path = Path(db_name)

readonly = True
readonly = False

create=False
# create=True

the_db = Db.open(path, readonly=readonly, create=create)
env = the_db.env

In [ ]:
# show lmdb env info

env.flags(), env.path(), env.info()

In [43]:
env.close()

In [4]:
# list all key-value pairs in the db, using lmdb's env

with env.begin(
    write=False,
    # buffers= ?
) as txn:
    with txn.cursor() as cur:
        for key, value in cur.iternext():
            # if key.startswith(b'r:'):
            #     print(key, value)
            print(key, value)


b'f:\x00*\x00\x01\x00\x01' b'\x08\x01\x10*\x18\x01(\x018\xb2\xf2\x19B abcd1234abcd1234abcd1234abcd1234'
b'fh:\xab\xcd\x124\xab\xcd\x124\xab\xcd\x124\xab\xcd\x124' b'\x00*\x00\x01\x00\x01'
b'r:\x00*' b'\x08\x01\x10*\x1a\x11C:\\some\\start\\dir" run from jupyter for development*\x05hoppa5\x9a\x99\x99?=\x9a\x99\x99?B\x0binitializedP\x08X\x18b\x0b\n\x04fld1\x12\x03blab\x16\n\x04fld2\x12\x0esan 64\r\nder 19'


In [9]:
CompositeId.bytes_to_bytes(b'\x00*\x01\x01\x00\x01')


(b'\x00*', b'\x01\x01', b'\x00\x01')

In [19]:
# REMOVE a key-value pair from db

key = b'\x00*'
with env.begin(write=True) as txn:
    value = txn.pop(key)

value


b'\x08\x01\x10*\x1a\x11C:\\some\\start\\dir" run from jupyter for development*\x05hoppa5\x9a\x99\x99?=\x9a\x99\x99?B\x0binitializedP\x08X\x18b\x0b\n\x04fld1\x12\x03blab\x16\n\x04fld2\x12\x0esan 64\r\nder 19'

In [5]:
# create a Run obj

run42 = Run(
    id=Id(42),
    path=Path('C:\\some\\start\\dir'),
    description='run from jupyter for development',
    platform='lekker plat',
    start_time=1.2,
    end_time=2.4,
    duration=1.2,
    num_dirs=8,
    num_files=24,
    extra={'fld1': 'bla', 'fld2': 'san 64\r\nder 19'},
    status='initialized',
)

In [6]:
run42.id, run42.path, run42.root_dir, run42.platform

(Id(42), WindowsPath('C:/some/start/dir'), None, 'lekker plat')

In [7]:
run42.platform = "hoppa"

In [8]:
# create a Dir obj

dir1 = Dir(
    run = run42,
    id = CompositeId(run42.id),
    name='dir',
    path=Path(''),
    path_repr='dir',
    timestamp=12345.6,
)

In [9]:
dir1.id, dir1.id.to_hex()

(CompositeId(42, 1), '002a0001')

In [10]:
run42.root_dir = dir1

In [11]:
# create a File obj

file1 = File(
    run = run42,
    id = CompositeId(dir1.id),
    name = 'some file',
    path=Path(''),
    dir=dir1,
    length=424242,
)

In [12]:
file1.id, file1.id.to_hex()

(CompositeId(42, 1, 1), '002a00010001')

In [13]:
file1.hash = 'abcd1234' * 4

In [21]:
# Store Run obj

the_db.put_run(run42)

Stored run 42


In [20]:
# Restore Run obj from db

run_id = Id(42)
rrun42 = the_db.get_run(run_id)
rrun42.id


Id(42)

In [22]:
rrun42.id.to_hex() == run42.id.to_hex()

True

In [23]:
run42.root_dir, rrun42.root_dir


(Dir(run=Run(id=Id(42), path=WindowsPath('C:/some/start/dir'), description='run from jupyter for development', platform='hoppa', start_time=1.2, end_time=2.4, duration=1.2, root_dir=..., extra={'fld1': 'bla', 'fld2': 'san 64\r\nder 19'}, status='initialized', num_dirs=8, num_files=24, error=None), id=CompositeId(42, 1), name='dir', path=WindowsPath('.'), path_repr='dir', timestamp=12345.6, parent=None, num_files=-1, num_dirs=-1, file_ids=[], dir_ids=[], file_hashes=[], dir_hashes=[], files_hash='', dirs_hash='', all_hash=''),
 None)

In [15]:
# store File obj

the_db.put_file(file1)
# len(file1.hash)

Stored file (42, 1, 1)


In [28]:
p = Path('to\\some\\dirc')
p, str(p)

(WindowsPath('to/some/dirc'), 'to\\some\\dirc')